# `evaluation.py` — Mesure de la qualité du système

> **MODULE FOURNI — NE PAS MODIFIER**

## L'exactitude (*accuracy*) est INTERDITE comme mesure principale

Si 80 % des courriels sont légitimes, un système qui répond toujours « ham »
obtient **80 % d'exactitude et ne détecte aucune attaque**.

## Le cas particulier du verdict « SUSPECT »

Le système peut répondre `suspect` (quarantaine), qui n'est ni une livraison ni
un blocage. Or les étiquettes du jeu de données n'ont que **trois** valeurs :
`ham`, `spam`, `phishing`.

On calcule donc les mesures sous **DEUX politiques** :

| Politique | Traitement de la quarantaine | Lecture |
|---|---|---|
| **PRUDENTE** | `suspect` → `phishing` | la quarantaine compte comme un blocage |
| **PERMISSIVE** | `suspect` → `ham` | la quarantaine compte comme une livraison |

Les mêmes prédictions donnent alors deux résultats différents. Ce n'est pas un
défaut : c'est la démonstration que « la performance » d'un système dépend de ce
qu'on décide de faire des **cas incertains**.

## Imports

`pandas` pour la matrice de confusion, `scikit-learn` pour les mesures par classe.
Les deux sont déjà installés dans Google Colab.

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

---
## Les étiquettes et les verdicts

`CLASSES` est la vérité terrain (3 valeurs, colonne `label` du CSV).
`VERDICTS` est ce que le système peut répondre (4 valeurs, `suspect` en plus).

C'est tout l'enjeu du module : **une colonne de plus du côté des prédictions**.

In [ ]:
CLASSES = ["ham", "spam", "phishing"]
VERDICTS = ["ham", "spam", "suspect", "phishing"]

---
## `COUTS` — toutes les erreurs ne se valent pas

Coût relatif des erreurs. À lire ainsi : bloquer un courriel légitime comme
hameçonnage est **10 fois** plus grave que laisser passer une publicité.

| Vraie classe ↓ / Verdict → | `ham` | `spam` | `suspect` | `phishing` |
|---|---|---|---|---|
| `ham` | — | 5 | 3 | **10** |
| `spam` | 1 | — | 1 | 2 |
| `phishing` | **8** | 3 | 2 | — |

Les deux cases en gras sont le compromis central du projet : il n'existe aucun
réglage qui les minimise toutes les deux à la fois.

`POLITIQUES` associe chaque politique à la classe qui remplace `suspect`.

In [ ]:
# Coût relatif des erreurs. À lire ainsi : bloquer un courriel légitime
# comme hameçonnage est 10 fois plus grave que laisser passer une pub.
COUTS = {
    ("ham", "spam"): 5,
    ("ham", "suspect"): 3,
    ("ham", "phishing"): 10,
    ("spam", "ham"): 1,
    ("spam", "suspect"): 1,
    ("spam", "phishing"): 2,
    ("phishing", "ham"): 8,
    ("phishing", "suspect"): 2,
    ("phishing", "spam"): 3,
}

POLITIQUES = {
    "prudente": "phishing",     # suspect traité comme une menace
    "permissive": "ham",        # suspect traité comme livrable
}

---
## `matrice_confusion(reels, verdicts)`

Tableau **3 lignes** (vraie classe) × **4 colonnes** (verdict rendu).

Le `reindex()` force l'ordre des lignes et des colonnes et ajoute les cases
vides : sans lui, une classe jamais prédite disparaîtrait du tableau et la
matrice changerait de forme d'une exécution à l'autre.

In [ ]:
def matrice_confusion(reels, verdicts):
    df = pd.DataFrame({"reel": list(reels), "verdict": list(verdicts)})
    m = pd.crosstab(df["reel"], df["verdict"])
    m = m.reindex(index=CLASSES, columns=VERDICTS, fill_value=0)
    m.index = [f"reel:{c}" for c in CLASSES]
    m.columns = [f"pred:{v}" for v in VERDICTS]
    return m

---
## `cout_total(reels, verdicts)`

Somme pondérée des erreurs, selon le tableau `COUTS`. **Plus bas = mieux.**

Les verdicts corrects (`r == v`) ne coûtent rien et sont ignorés. Un seul
nombre pour comparer deux réglages de seuils : c'est la mesure la plus directe
du compromis que vous avez choisi.

In [ ]:
def cout_total(reels, verdicts):
    return sum(COUTS.get((r, v), 0) for r, v in zip(reels, verdicts) if r != v)

---
## `appliquer_politique(verdicts, politique)`

Remplace « suspect » par la classe imposée par la politique, pour pouvoir
comparer des prédictions à 4 valeurs avec des étiquettes à 3 valeurs.

In [ ]:
def appliquer_politique(verdicts, politique="prudente"):
    cible = POLITIQUES[politique]
    return [cible if v == "suspect" else v for v in verdicts]

---
## `rapport(reels, verdicts, titre)`

Bilan complet, et **la fonction que vous appellerez le plus souvent**. Elle affiche :

1. la matrice de confusion des verdicts **bruts** (avec la colonne `suspect`) ;
2. le nombre de courriels mis en quarantaine ;
3. précision, rappel et F1 **par classe**, sous les deux politiques ;
4. le nombre d'erreurs et le coût total pondéré.

Elle **renvoie** un dictionnaire de mesures clés : c'est lui que vous passerez à
`comparer()` pour construire le tableau comparatif du rapport.

`zero_division=0` évite un plantage quand une classe n'est jamais prédite —
cas fréquent au début, quand tout ressort en `ham`.

In [ ]:
def rapport(reels, verdicts, titre="Resultats"):
    reels = list(reels)
    verdicts = list(verdicts)

    print("\n" + "=" * 66)
    print(f"  {titre}")
    print("=" * 66)

    print("\nMatrice de confusion (verdicts bruts)")
    print(matrice_confusion(reels, verdicts).to_string())

    n_suspects = verdicts.count("suspect")
    print(f"\nMis en quarantaine : {n_suspects} / {len(verdicts)}")

    mesures = {"titre": titre, "quarantaine": n_suspects,
               "cout": cout_total(reels, verdicts),
               "erreurs": sum(1 for r, v in zip(reels, verdicts) if r != v)}

    for politique in POLITIQUES:
        convertis = appliquer_politique(verdicts, politique)
        print(f"\n--- Politique {politique.upper()} "
              f"(suspect -> {POLITIQUES[politique]}) ---")
        print(classification_report(reels, convertis, labels=CLASSES,
                                    zero_division=0, digits=3))
        detail = classification_report(reels, convertis, labels=CLASSES,
                                       output_dict=True, zero_division=0)
        mesures[f"f1_macro_{politique}"] = detail["macro avg"]["f1-score"]
        mesures[f"rappel_phishing_{politique}"] = detail["phishing"]["recall"]
        mesures[f"precision_phishing_{politique}"] = detail["phishing"]["precision"]
        mesures[f"rappel_ham_{politique}"] = detail["ham"]["recall"]

    print(f"Erreurs (verdicts bruts) : {mesures['erreurs']} / {len(reels)}")
    print(f"Cout total pondere       : {mesures['cout']}  (plus bas = mieux)")

    return mesures

---
## `montrer_erreurs(courriels, reels, verdicts, n=8)`

Affiche les courriels mal classés — **la matière première de l'analyse
d'erreurs** exigée dans le rapport. Elle renvoie la liste complète des fautes,
même si elle n'en affiche que les `n` premières.

Les questions d'analyse portent précisément là-dessus : trois légitimes signalés
à tort, trois hameçonnages laissés passer, et ce qui leur manquait.

In [ ]:
def montrer_erreurs(courriels, reels, verdicts, n=8):
    fautes = [(c, r, v) for c, r, v in zip(courriels, reels, verdicts) if r != v]
    print(f"\n--- {len(fautes)} erreur(s), {min(n, len(fautes))} affichee(s) ---")
    for c, r, v in fautes[:n]:
        print(f"  reel={r:<9} verdict={v:<9} | {c.get('subject', '')[:48]}")
    return fautes

---
## `comparer(*resultats)`

Tableau comparatif des approches — **livrable noté du projet**.

On lui passe plusieurs dictionnaires renvoyés par `rapport()` :

```python
mesures_regles = rapport(reels, verdicts, "Regles Python")
mesures_texte  = rapport(reels, predictions_texte, "IA - texte")
comparer(mesures_regles, mesures_texte)
```

Chaque `titre` devient une ligne du tableau : c'est pourquoi il faut donner à
chaque appel de `rapport()` un titre distinct et parlant.

In [ ]:
def comparer(*resultats):
    tableau = pd.DataFrame(resultats).set_index("titre")
    print("\n" + "=" * 66)
    print("  COMPARAISON DES APPROCHES")
    print("=" * 66)
    print(tableau.round(3).to_string())
    return tableau